# Clase 5 — Pandas II: operaciones sobre DataFrames

**Python y Políticas Públicas**

---

## Contenidos
1. GroupBy y agregaciones
2. Merge y join
3. Concatenar DataFrames
4. Pivot tables y reshaping
5. Apply y funciones sobre columnas
6. Ejercicios

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Recreamos el dataset de la clase anterior
np.random.seed(42)
n = 200
provincias_lista = [
    "Buenos Aires", "Córdoba", "Santa Fe", "Mendoza", "Tucumán",
    "Salta", "Entre Ríos", "Chaco", "Misiones", "Corrientes"
]
programas_lista = ["AUH", "Progresar", "Potenciar Trabajo", "Alimentar", "Crédito Argenta"]

df = pd.DataFrame({
    "id_beneficiario": range(1001, 1001 + n),
    "provincia": np.random.choice(provincias_lista, n),
    "programa": np.random.choice(programas_lista, n),
    "edad": np.random.randint(18, 65, n),
    "genero": np.random.choice(["F", "M", "X"], n, p=[0.58, 0.40, 0.02]),
    "ingreso_mensual": np.random.lognormal(10.2, 0.6, n).round(0),
    "anios_educacion": np.random.randint(6, 18, n),
    "tiene_empleo_formal": np.random.choice([True, False], n, p=[0.35, 0.65]),
    "monto_transferencia": np.random.choice([18000, 25000, 42000, 55000, 10000], n),
})

print(f"Dataset: {df.shape[0]} filas × {df.shape[1]} columnas")
df.head(3)

---
## 1. GroupBy y agregaciones

`groupby` es una de las operaciones más poderosas de pandas. Permite dividir los datos en grupos, aplicar una función a cada grupo y combinar los resultados.

In [ ]:
# Promedio de ingreso y monto por programa
df.groupby('programa')[['ingreso_mensual', 'monto_transferencia']].mean().round(0)

In [ ]:
# Múltiples funciones de agregación
resumen = df.groupby('provincia').agg(
    cantidad=("id_beneficiario", "count"),
    ingreso_promedio=("ingreso_mensual", "mean"),
    edad_promedio=("edad", "mean"),
    pct_empleo_formal=("tiene_empleo_formal", "mean"),
).round(1)

resumen['pct_empleo_formal'] = (resumen['pct_empleo_formal'] * 100).round(1)
resumen.sort_values('cantidad', ascending=False)

In [ ]:
# Agrupar por múltiples columnas
por_programa_genero = df.groupby(['programa', 'genero'])['ingreso_mensual'].mean().round(0)
print(por_programa_genero.unstack())  # unstack convierte el último nivel en columnas

In [ ]:
# transform: agrega el resultado al DataFrame original (sin reducir filas)
df['ingreso_promedio_provincial'] = df.groupby('provincia')['ingreso_mensual'].transform('mean')
df['ingreso_relativo'] = (df['ingreso_mensual'] / df['ingreso_promedio_provincial']).round(2)

df[['provincia', 'ingreso_mensual', 'ingreso_promedio_provincial', 'ingreso_relativo']].head(8)

---
## 2. Merge y join

`merge` combina dos DataFrames usando una o más columnas clave, como un `JOIN` en SQL.

In [ ]:
# DataFrame de indicadores provinciales (tabla de referencia)
indicadores = pd.DataFrame({
    "provincia": provincias_lista,
    "pobreza_pct": [42.3, 38.1, 39.7, 34.2, 48.6, 52.1, 37.5, 58.3, 44.1, 46.8],
    "region": ["Pampeana", "Pampeana", "Pampeana", "Cuyo", "NOA",
               "NOA", "Pampeana", "NEA", "NEA", "NEA"],
    "pbi_pc_miles": [9.2, 10.1, 9.8, 8.5, 6.8, 6.1, 7.9, 5.5, 5.8, 5.9],
})
indicadores

In [ ]:
# Inner join: solo filas con clave en ambas tablas
df_enriquecido = df.merge(indicadores, on='provincia', how='left')

print(f"Filas antes del merge: {len(df)}")
print(f"Filas después del merge: {len(df_enriquecido)}")

df_enriquecido[['provincia', 'programa', 'ingreso_mensual', 'pobreza_pct', 'region']].head()

In [ ]:
# Tipos de merge
# how='inner'  → solo filas con coincidencia en ambas tablas (por defecto)
# how='left'   → todas las filas de la izquierda, NaN donde no hay match a la derecha
# how='right'  → todas las filas de la derecha
# how='outer'  → todas las filas de ambas tablas

# Merge con nombres de columna distintos
df_alt = pd.DataFrame({'prov': ['Buenos Aires', 'Córdoba'], 'codigo': [1, 2]})
# df.merge(df_alt, left_on='provincia', right_on='prov')

print("Ver comentarios en el código para los tipos de merge")

---
## 3. Concatenar DataFrames

In [ ]:
# Datos de tres trimestres (mismo formato, distintos períodos)
q1 = pd.DataFrame({'trimestre': ['Q1'] * 3, 'provincia': ['BA', 'Cba', 'SF'], 'valor': [100, 95, 102]})
q2 = pd.DataFrame({'trimestre': ['Q2'] * 3, 'provincia': ['BA', 'Cba', 'SF'], 'valor': [105, 98, 108]})
q3 = pd.DataFrame({'trimestre': ['Q3'] * 3, 'provincia': ['BA', 'Cba', 'SF'], 'valor': [110, 103, 112]})

# Concatenar verticalmente (apilar)
todos = pd.concat([q1, q2, q3], ignore_index=True)
print("Concatenación vertical:")
print(todos)

# Uso típico: cargar múltiples archivos y apilarlos
# archivos = ['datos_2021.csv', 'datos_2022.csv', 'datos_2023.csv']
# df_total = pd.concat([pd.read_csv(f) for f in archivos], ignore_index=True)

---
## 4. Pivot tables y reshaping

In [ ]:
# Pivot table: resumen cruzado (como tabla dinámica de Excel)
pivot = df_enriquecido.pivot_table(
    values='monto_transferencia',
    index='region',
    columns='genero',
    aggfunc='mean',
    margins=True,  # fila/columna de totales
    margins_name='Total'
).round(0)

pivot

In [ ]:
# Tabla de conteos: beneficiarios por provincia y programa
tabla_cruzada = pd.crosstab(
    df['provincia'],
    df['programa'],
    margins=True
)
tabla_cruzada

In [ ]:
# melt: pasar de formato wide a long (útil para graficar series temporales)
wide = pd.DataFrame({
    'provincia': ['Buenos Aires', 'Córdoba', 'Santa Fe'],
    'pobreza_2021': [40.1, 36.2, 38.0],
    'pobreza_2022': [42.3, 38.1, 39.7],
    'pobreza_2023': [40.9, 37.4, 38.5],
})

long = wide.melt(
    id_vars='provincia',
    value_vars=['pobreza_2021', 'pobreza_2022', 'pobreza_2023'],
    var_name='anio',
    value_name='pobreza_pct'
)
long['anio'] = long['anio'].str.replace('pobreza_', '').astype(int)
print("Formato long (ideal para gráficos de series temporales):")
long.sort_values(['provincia', 'anio'])

---
## 5. Apply y funciones sobre columnas

In [ ]:
# apply: aplicar una función a cada fila o columna
def clasificar_ingreso(ingreso):
    if pd.isna(ingreso):
        return "Sin dato"
    elif ingreso < 50_000:
        return "Bajo"
    elif ingreso < 150_000:
        return "Medio"
    else:
        return "Alto"

df['nivel_ingreso'] = df['ingreso_mensual'].apply(clasificar_ingreso)
print(df['nivel_ingreso'].value_counts())

In [ ]:
# map: reemplazar valores usando un diccionario
mapa_regiones = {
    "Buenos Aires": "Pampeana", "Córdoba": "Pampeana", "Santa Fe": "Pampeana", "Entre Ríos": "Pampeana",
    "Mendoza": "Cuyo",
    "Tucumán": "NOA", "Salta": "NOA",
    "Chaco": "NEA", "Misiones": "NEA", "Corrientes": "NEA"
}
df['region'] = df['provincia'].map(mapa_regiones)
print(df['region'].value_counts())

In [ ]:
# apply sobre filas (axis=1): combinar múltiples columnas
def score_vulnerabilidad(row):
    score = 0
    if not row['tiene_empleo_formal']:
        score += 2
    if row['ingreso_mensual'] < 50_000:
        score += 2
    if row['anios_educacion'] < 10:
        score += 1
    return score

df['score_vulnerabilidad'] = df.apply(score_vulnerabilidad, axis=1)
print("Distribución del score de vulnerabilidad:")
print(df['score_vulnerabilidad'].value_counts().sort_index())

---
## 6. Ejercicios

### Ejercicio 1
Calculá el monto total de transferencias por región (usá la columna `region` creada con `map`). ¿Qué región concentra más recursos?

In [ ]:
# Tu solución aquí


### Ejercicio 2
Creá una pivot table que muestre el **ingreso mensual promedio** por `region` (filas) y `nivel_ingreso` (columnas). Incluí márgenes.

In [ ]:
# Tu solución aquí


### Ejercicio 3
Creá un DataFrame `metas` con las siguientes metas de cobertura por programa, y hacé un merge con un resumen del `df` que contenga la cantidad real de beneficiarios por programa. Calculá qué porcentaje de la meta se alcanzó.

```python
metas = pd.DataFrame({
    'programa': ['AUH', 'Progresar', 'Potenciar Trabajo', 'Alimentar', 'Crédito Argenta'],
    'meta_beneficiarios': [50, 45, 35, 40, 30]  # metas en el dataset simulado
})
```

In [ ]:
# Tu solución aquí
